# 04 — Modelagem de Anomalia

Este notebook demonstra o fluxo de modelagem de anomalia: carregar o `dataset_analitico.parquet`, treinar um detector (IsolationForest neste exemplo) e salvar as pontuações e o modelo em `data/interim/` e `models/` respectivamente.

In [ ]:
# Carrega dataset analítico
from src.config import INTERIM_DIR
import pandas as pd

df = pd.read_parquet(INTERIM_DIR / 'dataset_analitico.parquet')
print('Linhas:', len(df))
print('Colunas:', len(df.columns))
df.head()

## Rodando o detector de anomalias

As células abaixo mostram um exemplo mínimo para treinar um IsolationForest e gerar `dataset_analitico_com_scores.parquet`. Execute apenas quando os arquivos em `data/interim/` estiverem presentes.

In [ ]:
# Exemplo: treinar IsolationForest e salvar scores
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib
from src.config import INTERIM_DIR, MODELS_DIR

# Carrega o dataset analítico já preparado
import pandas as pd
df = pd.read_parquet(INTERIM_DIR / 'dataset_analitico.parquet')

# Seleciona colunas numéricas para o exemplo
X = df.select_dtypes(include=["number"]).fillna(0)

clf = IsolationForest(contamination=0.05, random_state=42)
clf.fit(X)
scores = clf.decision_function(X)

# Anomalia label: os 5% mais baixos (ajustar conforme config)
df['anomalia_score'] = scores
df['anomalia_label'] = (scores < np.percentile(scores, 5)).astype(int)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, MODELS_DIR / 'modelo_supervisionado_anomalia.joblib')

# Salva dataset com scores
out_path = INTERIM_DIR / 'dataset_analitico_com_scores.parquet'
df.to_parquet(out_path, index=False)
print('Scores salvos em:', out_path)
print('Modelo salvo em:', MODELS_DIR / 'modelo_supervisionado_anomalia.joblib')